# Search and Load NISAR GCOV Data with `asf_search` and `xarray`

<br>

[`asf_search`](https://github.com/asfadmin/Discovery-asf_search) is an open source Python package for searching SAR data archived at {abbr}`ASF (Alaska Satellite Facility)`. This notebook demonstrates how to search NISAR GCOV with `asf_search`, download them with `earthaccess`, and load them into `xarray` data structures.

<hr>

## Overview

1. [Prerequisites](asf_search_stream-prereqs)
1. [Search for data](asf_search_stream-search)
1. [Load a single GCOV product](asf_search_stream-with)
1. [Load multiple GCOV Products at once](asf_search_stream-open_many)
1. [Build a time series](asf_search_stream-ts)
1. [Close open datasets](asf_search_stream-close)
1. [Summary](asf_search_stream-summary)
1. [Resources and references](asf_search_stream-resources)

<hr>

(asf_search_stream-prereqs)=
## 1. Prerequisites
| Prerequisite | Importance | Notes |
| --- | --- | --- |
| [The software environment for this cookbook must be installed](https://github.com/ASFOpenSARlab/NISAR_GCOV_Cookbook/blob/main/notebooks/create_software_environment.ipynb) | Necessary | |
| [An Earthdata Login account](https://urs.earthdata.nasa.gov/) | Necessary | Required for downloading NISAR data |

- **Rough Notebook Time Estimate**: 10 minutes

<hr>

(asf_search_stream-search)=
## 3. Search for data

Use `asf_search` to find GCOV data.

### 3a. Perform an `asf_search.search()` to identify your desired product URLs

In [1]:
import os
import asf_search as asf
from datetime import datetime
from getpass import getpass

import warnings
warnings.filterwarnings(
    "ignore",
    message="Parsing dates involving a day of month without a year specified",
)

session = asf.ASFSession()

start_date = datetime(2025, 11, 22)
end_date = datetime(2026, 3, 5)
area_of_interest = "POLYGON((40.9131 12.3904,41.8891 12.3904,41.8891 13.2454,40.9131 13.2454,40.9131 12.3904))" # POINT or POLYGON as WKT (well-known-text)
pattern = r'^(?!.*QA_STATS).*'

opts=asf.ASFSearchOptions(**{
    "maxResults": 250,
    "intersectsWith": area_of_interest,
    "flightDirection": "ASCENDING",
    "start": start_date,
    "end": end_date,
    "processingLevel": [
        "GCOV"
    ],
    "dataset": [
        "NISAR"
    ],
    "productionConfiguration": [
        "PR"
    ],
    'session': session,
})

response = asf.search(opts=opts)
hdf5_urls = response.find_urls(extension='.h5', pattern=pattern, directAccess=False)
print(f"Found {len(hdf5_urls)} GCOV products:")
hdf5_urls

Found 7 GCOV products:


['https://nisar.asf.earthdatacloud.nasa.gov/NISAR/NISAR_L2_GCOV_BETA_V1/NISAR_L2_PR_GCOV_005_172_A_008_2005_DHDH_A_20251122T024618_20251122T024652_X05007_N_F_J_001/NISAR_L2_PR_GCOV_005_172_A_008_2005_DHDH_A_20251122T024618_20251122T024652_X05007_N_F_J_001.h5',
 'https://nisar.asf.earthdatacloud.nasa.gov/NISAR/NISAR_L2_GCOV_BETA_V1/NISAR_L2_PR_GCOV_005_172_A_008_2005_DHDH_A_20251122T024618_20251122T024652_X05009_N_F_J_001/NISAR_L2_PR_GCOV_005_172_A_008_2005_DHDH_A_20251122T024618_20251122T024652_X05009_N_F_J_001.h5',
 'https://nisar.asf.earthdatacloud.nasa.gov/NISAR/NISAR_L2_GCOV_BETA_V1/NISAR_L2_PR_GCOV_006_172_A_008_2005_DHDH_A_20251204T024618_20251204T024653_X05007_N_F_J_001/NISAR_L2_PR_GCOV_006_172_A_008_2005_DHDH_A_20251204T024618_20251204T024653_X05007_N_F_J_001.h5',
 'https://nisar.asf.earthdatacloud.nasa.gov/NISAR/NISAR_L2_GCOV_BETA_V1/NISAR_L2_PR_GCOV_006_172_A_008_2005_DHDH_A_20251204T024618_20251204T024653_X05009_N_F_J_001/NISAR_L2_PR_GCOV_006_172_A_008_2005_DHDH_A_20251204T0

### 3b. Retain only the URL for the most recent version of each product in the search results

Data is occasionally re-released with an updated version. Versions are recorded as a [Composite Release Identifier (CRID)](https://nisar-docs.asf.alaska.edu/naming-conventions/) in a product's filename. We can use the CRID to retain only the most recent version of each product in the list of URLs.

In [2]:
import re

pattern = re.compile(r"(NISAR_L2_PR_GCOV(?:_[^_]+){9})_(X\d{5})")

latest_version_dict = {}

for url in hdf5_urls:
    m = pattern.search(url)
    if not m:
        continue

    product, crid = m.groups()

    if product not in latest_version_dict or crid > latest_version_dict[product][0]:
        latest_version_dict[product] = (crid, url)

hdf5_urls = [i[1] for i in latest_version_dict.values()]
print(f"Retained {len(hdf5_urls)} GCOV products:")
hdf5_urls

Retained 5 GCOV products:


['https://nisar.asf.earthdatacloud.nasa.gov/NISAR/NISAR_L2_GCOV_BETA_V1/NISAR_L2_PR_GCOV_005_172_A_008_2005_DHDH_A_20251122T024618_20251122T024652_X05009_N_F_J_001/NISAR_L2_PR_GCOV_005_172_A_008_2005_DHDH_A_20251122T024618_20251122T024652_X05009_N_F_J_001.h5',
 'https://nisar.asf.earthdatacloud.nasa.gov/NISAR/NISAR_L2_GCOV_BETA_V1/NISAR_L2_PR_GCOV_006_172_A_008_2005_DHDH_A_20251204T024618_20251204T024653_X05009_N_F_J_001/NISAR_L2_PR_GCOV_006_172_A_008_2005_DHDH_A_20251204T024618_20251204T024653_X05009_N_F_J_001.h5',
 'https://nisar.asf.earthdatacloud.nasa.gov/NISAR/NISAR_L2_GCOV_BETA_V1/NISAR_L2_PR_GCOV_007_172_A_008_2005_DHDH_A_20251216T024619_20251216T024653_X05009_N_F_J_001/NISAR_L2_PR_GCOV_007_172_A_008_2005_DHDH_A_20251216T024619_20251216T024653_X05009_N_F_J_001.h5',
 'https://nisar.asf.earthdatacloud.nasa.gov/NISAR/NISAR_L2_GCOV_BETA_V1/NISAR_L2_PR_GCOV_008_172_A_008_2005_DHDH_A_20251228T024619_20251228T024654_X05009_N_F_J_001/NISAR_L2_PR_GCOV_008_172_A_008_2005_DHDH_A_20251228T0

### 3c. Download the HDF5 files locally

Since direct S3 access requires running in AWS `us-west-2`, we download the files via HTTPS using `earthaccess`.

In [ ]:
import earthaccess
from pathlib import Path

earthaccess.login()

local_dir = Path("data")
local_dir.mkdir(exist_ok=True)

local_files = earthaccess.download(hdf5_urls, local_path=str(local_dir))
print(f"Downloaded {len(local_files)} files to {local_dir}")
local_files

<hr>

(asf_search_stream-with)=
## 4. Load a single GCOV product using a Python `with` statement

### 4a. Open a local GCOV file and compute the mean of an HHHH spatial subset

In [ ]:
%%time
import xarray as xr
import rioxarray

local_file = local_files[0]

dt = xr.open_datatree(
    local_file, 
    engine="h5netcdf", 
    decode_timedelta=False, 
    chunks="auto",
    phony_dims="access"
)

frequencyA = dt["/science/LSAR/GCOV/grids/frequencyA"]
projection = frequencyA.projection.attrs['epsg_code'].item()
hhhh = frequencyA.HHHH
hhhh = hhhh.rio.write_crs(projection)

subset_hhhh = hhhh.rio.clip_box(
    minx=40.8463, miny=13.2553,
    maxx=40.8574, maxy=13.2684, 
    crs="EPSG:4326"
)

subset_hhhh_mean = subset_hhhh.mean().to_numpy().item()
print(f'subset_hhhh_mean: {subset_hhhh_mean}')

### 4b. View the `DataTree`

You can look at the `DataTree` outside the `with` block, but it only contains pointers to the data and the file is closed, so you can no longer access any of the data values to which it points

In [6]:
dt

<xarray.DataTree>
Group: /
│   Attributes:
│       Conventions:         CF-1.7
│       contact:             nisar-sds-ops@jpl.nasa.gov
│       institution:         NASA JPL
│       mission_name:        NISAR
│       reference_document:  D-102274 NISAR NASA SDS Product Specification Level-...
│       title:               NISAR L2 GCOV Product
└── Group: /science
    └── Group: /science/LSAR
        ├── Group: /science/LSAR/GCOV
        │   ├── Group: /science/LSAR/GCOV/grids
        │   │   ├── Group: /science/LSAR/GCOV/grids/frequencyA
        │   │   │       Dimensions:                (yCoordinates: 16704, xCoordinates: 17064,
        │   │   │                                   phony_dim_0: 2)
        │   │   │       Coordinates:
        │   │   │         * yCoordinates           (yCoordinates) float64 134kB 1.588e+06 ... 1.254e+06
        │   │   │         * xCoordinates           (xCoordinates) float64 137kB 6.041e+05 ... 9.454e+05
        │   │   │       Dimensions without coordinates: phony_dim_0
        │   │   │       Data variables:
        │   │   │           HHHH                   (yCoordinates, xCoordinates) float32 1GB dask.array<chunksize=(5632, 5632), meta=np.ndarray>
        │   │   │           HVHV                   (yCoordinates, xCoordinates) float32 1GB dask.array<chunksize=(5632, 5632), meta=np.ndarray>
        │   │   │           listOfCovarianceTerms  (phony_dim_0) |S4 8B dask.array<chunksize=(2,), meta=np.ndarray>
        │   │   │           listOfPolarizations    (phony_dim_0) |S2 4B dask.array<chunksize=(2,), meta=np.ndarray>
        │   │   │           mask                   (yCoordinates, xCoordinates) float32 1GB dask.array<chunksize=(5632, 5632), meta=np.ndarray>
        │   │   │           numberOfLooks          (yCoordinates, xCoordinates) float32 1GB dask.array<chunksize=(5632, 5632), meta=np.ndarray>
        │   │   │           numberOfSubSwaths      uint8 1B ...
        │   │   │           projection             uint32 4B ...
        │   │   │           rtcGammaToSigmaFactor  (yCoordinates, xCoordinates) float32 1GB dask.array<chunksize=(5632, 5632), meta=np.ndarray>
        │   │   │           xCoordinateSpacing     float64 8B ...
        │   │   │           yCoordinateSpacing     float64 8B ...
        │   │   └── Group: /science/LSAR/GCOV/grids/frequencyB
        │   │           Dimensions:                (yCoordinates: 4176, xCoordinates: 4266,
        │   │                                       phony_dim_1: 2)
        │   │           Coordinates:
        │   │             * yCoordinates           (yCoordinates) float64 33kB 1.588e+06 ... 1.254e+06
        │   │             * xCoordinates           (xCoordinates) float64 34kB 6.041e+05 ... 9.453e+05
        │   │           Dimensions without coordinates: phony_dim_1
        │   │           Data variables:
        │   │               HHHH                   (yCoordinates, xCoordinates) float32 71MB dask.array<chunksize=(4176, 4266), meta=np.ndarray>
        │   │               HVHV                   (yCoordinates, xCoordinates) float32 71MB dask.array<chunksize=(4176, 4266), meta=np.ndarray>
        │   │               listOfCovarianceTerms  (phony_dim_1) |S4 8B dask.array<chunksize=(2,), meta=np.ndarray>
        │   │               listOfPolarizations    (phony_dim_1) |S2 4B dask.array<chunksize=(2,), meta=np.ndarray>
        │   │               mask                   (yCoordinates, xCoordinates) float32 71MB dask.array<chunksize=(4176, 4266), meta=np.ndarray>
        │   │               numberOfLooks          (yCoordinates, xCoordinates) float32 71MB dask.array<chunksize=(4176, 4266), meta=np.ndarray>
        │   │               numberOfSubSwaths      uint8 1B ...
        │   │               projection             uint32 4B ...
        │   │               rtcGammaToSigmaFactor  (yCoordinates, xCoordinates) float32 71MB dask.array<chunksize=(4176, 4266), meta=np.ndarray>
        │   │               xCoordinateSpacing     float64 8B ...
    

### 4c. Try (and fail) to compute a value in the `DataTree`

The file was closed upon exiting the `with` block so trying to compute a value from the `DataTree` will fail with a `ValueError: I/O operation on closed file.`

In [7]:
dt.compute()

ValueError: I/O operation on closed file.

#### However, `subset_hhhh_mean` was computed and saved to memory inside the `with` block, so we can still see its value

In [8]:
subset_hhhh_mean

0.07500407099723816

<hr>

(asf_search_stream-open_many)=
## 5. Load multiple GCOV Products at once

:::{note} Lets speed things up a bit

If you are not working with all of the groups in an HDF5 file, there is no need to spend time loading the entire file, as was done in the example above.

For the remaining examples in this notebook, we will access only the Frequency A data, stored in the group `/science/LSAR/GCOV/grids/frequencyA`.
:::

### 5a. Open the `/science/LSAR/GCOV/grids/frequencyA` group from each local file

In [ ]:
%%time
import xarray as xr
import rioxarray

group_path = "/science/LSAR/GCOV/grids/frequencyA"

datatrees = [
    xr.open_datatree(
        f,
        engine="h5netcdf",
        decode_timedelta=False,
        phony_dims="access",
        chunks="auto",
        group=group_path,
    )
    for f in local_files
]

### 5b. Since the files remain open, we can access still access them and perform computations on their contents

Iterate through the `DataTrees`, calculating subset HHHH mean values

In [ ]:
import warnings

with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    for tree in datatrees:
        projection = tree.projection.attrs['epsg_code'].item()
        hhhh = tree.HHHH
        hhhh = hhhh.rio.write_crs(projection)
        
        subset_hhhh = hhhh.rio.clip_box(
            minx=40.8463, miny=13.2553,
            maxx=40.8574, maxy=13.2684, 
            crs="EPSG:4326"
        )
        subset_hhhh_mean = subset_hhhh.mean().to_numpy().item()
        print(subset_hhhh_mean)

<hr>

(asf_search_stream-ts)=
## 6. Stream a time series

Use the multiple files we opened in Step 5 to create a GCOV time series

### 6a. Define a function to extract datetimes for a `time` dimension

In [ ]:
import re
from datetime import datetime
from pathlib import Path

NISAR_TS_RE = re.compile(r"_(\d{8}T\d{6})_")

def nisar_start_time_from_path(file_path: str) -> datetime:
    name = Path(file_path).name
    m = NISAR_TS_RE.search(name)
    if not m:
        raise ValueError(f"No NISAR timestamp found in: {file_path}")
    return datetime.strptime(m.group(1), "%Y%m%dT%H%M%S")

### 6b. Create a list of datetimes for a `time` dimension

In [ ]:
dts = [nisar_start_time_from_path(f) for f in local_files]
dts

### 6c. Create an `xarray.DataArray` containing a `time` dimension for each open GCOV group

In [13]:
dataarrays = [
    tree.ds.assign_coords(time=dt).expand_dims(time=1)
    for dt, tree in zip(dts, datatrees)
]

dataarrays[0].time

<xarray.DataArray 'time' (time: 1)> Size: 8B
array(['2025-11-22T02:46:18.000000000'], dtype='datetime64[ns]')
Coordinates:
  * time     (time) datetime64[ns] 8B 2025-11-22T02:46:18

In [14]:
for da in dataarrays:
    print(da.dims)

FrozenMappingWarningOnValuesAccess({'time': 1, 'yCoordinates': 16704, 'xCoordinates': 17064, 'phony_dim_0': 2})
FrozenMappingWarningOnValuesAccess({'time': 1, 'yCoordinates': 16704, 'xCoordinates': 17064, 'phony_dim_0': 2})
FrozenMappingWarningOnValuesAccess({'time': 1, 'yCoordinates': 16704, 'xCoordinates': 17064, 'phony_dim_0': 2})
FrozenMappingWarningOnValuesAccess({'time': 1, 'yCoordinates': 16704, 'xCoordinates': 17064, 'phony_dim_0': 2})
FrozenMappingWarningOnValuesAccess({'time': 1, 'yCoordinates': 16704, 'xCoordinates': 17064, 'phony_dim_0': 2})


### 6d. Concatenate the `xarray.DataArray`s into a single `xarray.Dataset` time series 

In [15]:
ts = xr.concat(dataarrays, dim="time")
ts

<xarray.Dataset> Size: 29GB
Dimensions:                (time: 5, yCoordinates: 16704, xCoordinates: 17064,
                            phony_dim_0: 2)
Coordinates:
  * time                   (time) datetime64[ns] 40B 2025-11-22T02:46:18 ... ...
  * yCoordinates           (yCoordinates) float64 134kB 1.588e+06 ... 1.254e+06
  * xCoordinates           (xCoordinates) float64 137kB 6.041e+05 ... 9.454e+05
Dimensions without coordinates: phony_dim_0
Data variables:
    HHHH                   (time, yCoordinates, xCoordinates) float32 6GB dask.array<chunksize=(1, 5632, 5632), meta=np.ndarray>
    HVHV                   (time, yCoordinates, xCoordinates) float32 6GB dask.array<chunksize=(1, 5632, 5632), meta=np.ndarray>
    listOfCovarianceTerms  (time, phony_dim_0) |S4 40B dask.array<chunksize=(1, 2), meta=np.ndarray>
    listOfPolarizations    (time, phony_dim_0) |S2 20B dask.array<chunksize=(1, 2), meta=np.ndarray>
    mask                   (time, yCoordinates, xCoordinates) float32 6GB dask.array<chunksize=(1, 5632, 5632), meta=np.ndarray>
    numberOfLooks          (time, yCoordinates, xCoordinates) float32 6GB dask.array<chunksize=(1, 5632, 5632), meta=np.ndarray>
    numberOfSubSwaths      (time) uint8 5B 1 1 1 1 1
    projection             (time) uint32 20B 32637 32637 32637 32637 32637
    rtcGammaToSigmaFactor  (time, yCoordinates, xCoordinates) float32 6GB dask.array<chunksize=(1, 5632, 5632), meta=np.ndarray>
    xCoordinateSpacing     (time) float64 40B 20.0 20.0 20.0 20.0 20.0
    yCoordinateSpacing     (time) float64 40B -20.0 -20.0 -20.0 -20.0 -20.0

<hr>

(asf_search_stream-close)=
## 7. Close open datasets

Close the datatrees when finished to free resources.

### 7a. Close the datatrees

In [ ]:
for tree in datatrees:
    tree.close()
dt.close()

<hr>

(asf_search_stream-summary)=
## 8. Summary
You now have the tools and knowledge that you need to search with `asf_search`, generate temporary S3 bucket credentials from an Earthdata Login Bearer Token, stream data from S3 with `s3fs`, and load them into `xarray` data structures.

<hr>

(asf_search_stream-resources)=
## 9. Resources and references
 - [asf_search](https://github.com/asfadmin/Discovery-asf_search)
 - [NISAR Data User Guide](https://nisar-docs.asf.alaska.edu/aws-s3-access/)
 
**Author:** Alex Lewandowski